### import dep.

In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance,PayloadSchemaType,PointStruct,SparseVectorParams,Document,Prefetch,FusionQuery
from qdrant_client import models

import pandas as pd
from groq import Groq
import fastembed


c:\Users\Loq\Documents\CRAP\end to end aibootcamp\code\handsON\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
qdrant_client=QdrantClient(url="http://localhost:6333")

### Create a Qdrant Collection for Hybrid Search

In [3]:
qdrant_client.create_collection(
    collection_name="amazon-items-collection-01-hybrid-search",
    vectors_config={
        "all-MiniLM-L6-v2": VectorParams(size=384, distance=Distance.COSINE),
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams(modifier=models.Modifier.IDF)
    }

)

True

In [4]:
qdrant_client.create_payload_index(
    collection_name="amazon-items-collection-01-hybrid-search",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding Functions

In [5]:
import json
import os
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

HF_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
HF_API_TOKEN = os.environ.get('HF_API_TOKEN')  # set your Hugging Face token, e.g., os.environ.get('HF_API_TOKEN')


def _mean_pool_embedding(raw_embedding):
    if not raw_embedding:
        raise ValueError("Hugging Face API returned an empty embedding.")

    if isinstance(raw_embedding[0], (int, float)):
        return [float(value) for value in raw_embedding]

    if isinstance(raw_embedding[0], list):
        token_count = len(raw_embedding)
        vector_size = len(raw_embedding[0])
        pooled = [0.0] * vector_size

        for token_vector in raw_embedding:
            if len(token_vector) != vector_size:
                raise ValueError("Inconsistent token vector dimensions in HF embedding response.")
            for idx, value in enumerate(token_vector):
                pooled[idx] += float(value)

        return [value / token_count for value in pooled]

    raise ValueError("Unexpected Hugging Face embedding response format.")


def get_embedding(text, model_name: str | None = None):
    selected_model = model_name or HF_EMBEDDING_MODEL
    endpoint = (
        "https://router.huggingface.co/hf-inference/models/"
        f"{selected_model}/pipeline/feature-extraction"
    )
    payload = json.dumps({"inputs": text, "normalize": True}).encode("utf-8")

    headers = {"Content-Type": "application/json"}
    if HF_API_TOKEN:
        headers["Authorization"] = f"Bearer {HF_API_TOKEN}"

    request = Request(endpoint, data=payload, headers=headers, method="POST")

    try:
        with urlopen(request, timeout=60) as response:
            response_data = json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        message = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Hugging Face embedding API request failed ({exc.code}): {message}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(f"Could not reach Hugging Face embedding API: {exc}") from exc

    if isinstance(response_data, dict) and response_data.get("error"):
        raise RuntimeError(f"Hugging Face embedding API error: {response_data['error']}")

    if isinstance(response_data, list) and len(response_data) == 1:
        return _mean_pool_embedding(response_data[0])

    return _mean_pool_embedding(response_data)


def get_embedding_batch(text_list, batch_size=100, model_name: str | None = None):
    if not isinstance(text_list, list):
        raise ValueError("text_list must be a list of strings")
    if len(text_list) == 0:
        return []

    embeddings = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i : i + batch_size]
        endpoint = (
            "https://router.huggingface.co/hf-inference/models/"
            f"{(model_name or HF_EMBEDDING_MODEL)}/pipeline/feature-extraction"
        )
        payload = json.dumps({"inputs": batch, "normalize": True}).encode("utf-8")

        headers = {"Content-Type": "application/json"}
        if HF_API_TOKEN:
            headers["Authorization"] = f"Bearer {HF_API_TOKEN}"

        request = Request(endpoint, data=payload, headers=headers, method="POST")

        try:
            with urlopen(request, timeout=120) as response:
                response_data = json.loads(response.read().decode("utf-8"))
        except HTTPError as exc:
            message = exc.read().decode("utf-8", errors="replace")
            raise RuntimeError(
                f"Hugging Face embedding API request failed ({exc.code}): {message}"
            ) from exc
        except URLError as exc:
            raise RuntimeError(f"Could not reach Hugging Face embedding API: {exc}") from exc

        if isinstance(response_data, dict) and response_data.get("error"):
            raise RuntimeError(f"Hugging Face embedding API error: {response_data['error']}")

        if not isinstance(response_data, list):
            raise ValueError("Unexpected response format from HF API for batch embedding")

        for raw in response_data:
            embeddings.append(_mean_pool_embedding(raw))

    return embeddings


# # Test for 1,000 entries
# if __name__ == "__main__":
#     sample_texts = [f"Test sentence {i}" for i in range(1000)]
#     print("Sending batch embedding request for 1000 sentences...")
#     result = get_embedding_batch(sample_texts, batch_size=100)
#     assert len(result) == 1000, f"expected 1000 embeddings, got {len(result)}"
#     assert all(len(v) > 0 for v in result), "some embeddings were empty"
#     print("Finished test: 1000 embeddings returned successfully")

### Process and Embed Amazon items Data

In [7]:
df_items=pd.read_json("./../../data/meta_CDs_and_Vinyl_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [8]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Digital Music,Queen II[LP],4.7,1835,[],[Queen II is the second studio album from the ...,29.33,[{'thumb': 'https://m.media-amazon.com/images/...,[],Queen Format: Vinyl,"[CDs & Vinyl, Rock]","{'Language': 'English', 'Product Dimensions': ...",B0BDFW4VRB,NaN
1,Digital Music,The Elephants of Mars Limited Orange,4.7,125,[],"[It moves, it swings, it rocks!Satriani and hi...",26.98,[{'thumb': 'https://m.media-amazon.com/images/...,[],Joe Satriani Format: Vinyl,"[CDs & Vinyl, Rock]",{'Product Dimensions': '12 x 12.3 x 0.4 inches...,B09Q98326Z,NaN
2,Digital Music,The Love Invention Heavyweight Black,4.5,158,[],[Alison Goldfrapp has set a towering bar for B...,20.49,[{'thumb': 'https://m.media-amazon.com/images/...,[],Alison Goldfrapp (Artist) Format: Vinyl,"[CDs & Vinyl, Dance & Electronic, Electronica]","{'Language': 'English', 'Product Dimensions': ...",B0BYHFG53W,NaN
3,Digital Music,Now That's What I Call A Love Song / Various,4.5,120,[],[NOW Music is proud to present 80 of the great...,25.68,[{'thumb': 'https://m.media-amazon.com/images/...,[],Various Artists (Artist) Format: Audio CD,"[CDs & Vinyl, Rock]",{'Package Dimensions': '5.51 x 4.92 x 0.51 inc...,B0BS1SHT91,NaN
4,Digital Music,SZNZ: Summer,4.4,120,[],"[SZNZ: Summer, the second part in Weezer's fou...",8.79,[{'thumb': 'https://m.media-amazon.com/images/...,[],Weezer Format: Audio CD,"[CDs & Vinyl, Indie & Alternative, Indie & Lo-Fi]","{'Language': 'English', 'Product Dimensions': ...",B0B2ZRV4FD,NaN


In [9]:
len(df_items)

1000

In [10]:
df_items["features"], df_items["description"] = df_items["description"], df_items["features"]

In [11]:
def preprocess_features(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [12]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [13]:
df_items["description"] = df_items.apply(preprocess_features, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [14]:
data_to_embed=df_items[["description", "image","rating_number","price","average_rating","parent_asin"]].to_dict(orient="records")

In [15]:
data_to_embed 

[{'description': 'Queen II[LP] Queen II is the second studio album from the iconic British rock band. Originally released in 1974, it reached #5 on the U.K. albums chart. This reissue is pressed on 180-gram vinyl and sourced from the original master tapes.',
  'image': 'https://m.media-amazon.com/images/I/41wmg-DUsDL.jpg',
  'rating_number': 1835,
  'price': 29.33,
  'average_rating': 4.7,
  'parent_asin': 'B0BDFW4VRB'},
 {'description': 'The Elephants of Mars Limited Orange It moves, it swings, it rocks!Satriani and his touring band, who all recorded remotely in separate areas of the world during lockdown, deliver an album-length journey that never dulls. The Elephants of Mars crackles with an exciting new energy, briskly traveling through stylistic roads that feel freshly updated, viewed through new eyes. The guitarist had an aim to create a “new standard” when it came to crafting an instrumental guitar record. He strove to openly challenge himself to move away from what he describes

In [16]:
text_to_embed = [item["description"] for item in data_to_embed]

In [19]:
embeddings = get_embedding_batch(text_to_embed, batch_size=100)

In [22]:
len(embeddings)

1000

In [24]:
len(embeddings[0])

384

In [25]:
pointstructs=[]
i=1
for embedding,data in zip(embeddings,data_to_embed):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "all-MiniLM-L6-v2": embedding,
                "bm25":Document(
                    text=data["description"],
                    model="qdrant/bm25"
                )
            },
            payload=data
        )
    )
    i+=1

In [27]:
qdrant_client.upsert(
    collection_name="amazon-items-collection-01-hybrid-search",
    points=pointstructs[0:500],
    wait=True
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [28]:
qdrant_client.upsert(
    collection_name="amazon-items-collection-01-hybrid-search",
    points=pointstructs[500:],
    wait=True
)

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

### Hybrid Retrieval

In [31]:
def retrieve_data(query,qdrant_client, top_k=5):

    query_embedding=get_embedding(query)

    search_result=qdrant_client.query_points(
        collection_name="amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="all-MiniLM-L6-v2",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=top_k,
    )

    retrieved_context_ids=[]
    retrieved_context=[]
    similarity_scores=[]
    retrieved_context_ratings=[]
    for search_result in search_result.points:
        retrieved_context_ids.append(search_result.payload["parent_asin"])
        retrieved_context.append(search_result.payload["description"])
        retrieved_context_ratings.append(search_result.payload["average_rating"])
        similarity_scores.append(search_result.score)


    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }

In [32]:
results=retrieve_data("rock music vinyl with good ratings", qdrant_client, top_k=20)

In [33]:
results

{'retrieved_context_ids': ['B0B4T2XKWZ',
  'B09XZ2RP29',
  'B0B9GZ98CZ',
  'B0B4GF4YVK',
  'B00L2QTQCE',
  'B0BX4HLCQQ',
  'B09SBQQYWZ',
  'B09SP39L9H',
  'B09NS1VZJF',
  'B00NBXVL9M',
  'B00VKZVPOY',
  'B0B3M8PDRX',
  'B09TX45B7J',
  'B0BVLC5KPC',
  'B0B1LYZRY2',
  'B09V1PRRP1',
  'B09QJZL4XM',
  'B09PRMQP5C',
  'B09X3NHJPS',
  'B0BHJH1WY1'],
 'retrieved_context': ['Winter Wonderland / Various Black Winter Wonderland / Various - 140-Gram Black Vinyl',
  'Live From The Astroturf Limited & Numbered Apricot Limited apricot colored vinyl LP + DVD. By 1974, the Alice Cooper group had risen to the upper echelon of rock stardom... and then, parted ways. In October 2015, over 40 years later, record store owner and superfan Chris Penn convinced the original line-up to reunite for a very special performance at his record store Good Records in Dallas, Texas. Alice Cooper (vocals), Michael Bruce (guitar), Dennis Dunaway (bass) and Neal Smith (drums) were joined on stage by Alice\'s current guitar